# bikem_hgb_hourcyc: HistGradientBoostingRegressor

설정:
- 대상 스테이션: `ST-17`
- 2024 데이터를 시간순으로 `train/valid` 분할
- test: `TPSS 2025`
- 모델: `HistGradientBoostingRegressor`

사용 피처:
- `hour_sin`
- `hour_cos`
- `month`
- `dayofweek`
- `is_weekend`
- `is_holiday`
- `outflow_now`
- `inflow_now`
- `netflow_now`
- `기온(°C)`
- `습도(%)`
- `적설(cm)`
- `outflow_now_lag1`
- `outflow_now_lag24`
- `inflow_now_lag1`
- `inflow_now_lag24`


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path('../Data')
TARGET_STATION_ID = 'ST-17'
RACK_COUNT = 23
TARGET_FILL = int(round(RACK_COUNT * 0.75))
VALID_RATIO = 0.2


In [2]:
TPSS_COLUMNS = [
    'date_key', 'agg_type', 'hour_key', 'start_station_id', 'start_station_name',
    'end_station_id', 'end_station_name', 'ride_count', 'duration_min', 'distance_m', 'source_file'
]


def load_tpss(filename: str, year_value: int) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / filename, encoding='utf-8-sig')
    df.columns = TPSS_COLUMNS
    df['year'] = year_value
    df['date'] = pd.to_datetime(df['date_key'].astype(str), format='%Y%m%d', errors='coerce')
    hour_text = df['hour_key'].astype(str).str.replace('.0', '', regex=False).str.zfill(4)
    df['hour'] = pd.to_numeric(hour_text.str[:2], errors='coerce').fillna(0).astype(int)
    df['ride_count'] = pd.to_numeric(df['ride_count'], errors='coerce').fillna(0)
    return df


def load_weather(filename: str) -> pd.DataFrame:
    weather = pd.read_csv(DATA_DIR / filename, encoding='utf-8-sig')
    weather['timestamp'] = pd.to_datetime(weather['일시'], errors='coerce')
    for col in ['기온(°C)', '습도(%)', '적설(cm)']:
        weather[col] = pd.to_numeric(weather[col], errors='coerce')
    weather['적설(cm)'] = weather['적설(cm)'].fillna(0)
    return weather[['timestamp', '기온(°C)', '습도(%)', '적설(cm)']]


tpss_2024 = load_tpss('tpss_bcycl_od_statnhm_2024_seodaemun_merged.csv', 2024)
tpss_2025 = load_tpss('tpss_bcycl_od_statnhm_2025_seodaemun_merged.csv', 2025)
tpss = pd.concat([tpss_2024, tpss_2025], ignore_index=True)

weather_2024 = load_weather('OBS_ASOS_TIM_2024.csv')
weather_2025 = load_weather('OBS_ASOS_TIM_2025_with_DI.csv')
weather = pd.concat([weather_2024, weather_2025], ignore_index=True)


In [3]:
outflow = (
    tpss[tpss['start_station_id'].astype(str).str.strip() == TARGET_STATION_ID]
    .groupby(['year', 'date', 'hour'], as_index=False)['ride_count']
    .sum()
    .rename(columns={'ride_count': 'outflow_now'})
)

inflow = (
    tpss[tpss['end_station_id'].astype(str).str.strip() == TARGET_STATION_ID]
    .groupby(['year', 'date', 'hour'], as_index=False)['ride_count']
    .sum()
    .rename(columns={'ride_count': 'inflow_now'})
)

station_hourly = (
    outflow.merge(inflow, on=['year', 'date', 'hour'], how='outer')
    .fillna(0)
    .sort_values(['year', 'date', 'hour'])
    .reset_index(drop=True)
)
station_hourly['timestamp'] = station_hourly['date'] + pd.to_timedelta(station_hourly['hour'], unit='h')
station_hourly['month'] = station_hourly['date'].dt.month
station_hourly['dayofweek'] = station_hourly['date'].dt.dayofweek
station_hourly['is_weekend'] = (station_hourly['dayofweek'] >= 5).astype(int)
station_hourly['netflow_now'] = station_hourly['inflow_now'] - station_hourly['outflow_now']
station_hourly['hour_sin'] = np.sin(2 * np.pi * station_hourly['hour'] / 24)
station_hourly['hour_cos'] = np.cos(2 * np.pi * station_hourly['hour'] / 24)
station_hourly = station_hourly.merge(weather, on='timestamp', how='left')
station_hourly[['기온(°C)', '습도(%)', '적설(cm)']] = station_hourly[['기온(°C)', '습도(%)', '적설(cm)']].ffill().bfill()


In [4]:
HOLIDAYS = {
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12',
    '2024-03-01', '2024-04-10', '2024-05-05', '2024-05-06', '2024-05-15',
    '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18',
    '2024-10-01', '2024-10-03', '2024-10-09', '2024-12-25',
    '2025-01-01', '2025-01-28', '2025-01-29', '2025-01-30',
    '2025-03-01', '2025-03-03', '2025-05-05', '2025-05-06',
    '2025-06-03', '2025-06-06', '2025-08-15',
    '2025-10-01', '2025-10-03', '2025-10-05', '2025-10-06', '2025-10-07', '2025-10-08', '2025-10-09',
    '2025-12-25',
}
station_hourly['date_str'] = station_hourly['date'].dt.strftime('%Y-%m-%d')
station_hourly['is_holiday'] = station_hourly['date_str'].isin(HOLIDAYS).astype(int)

remaining_values = []
current_remaining = RACK_COUNT
previous_year = None
previous_date = None

for row in station_hourly.itertuples(index=False):
    if previous_year != row.year:
        current_remaining = RACK_COUNT
    elif previous_date is not None and row.date.date() != previous_date:
        current_remaining = RACK_COUNT

    current_remaining = current_remaining - row.outflow_now + row.inflow_now
    current_remaining = int(max(0, min(RACK_COUNT, round(current_remaining))))
    remaining_values.append(current_remaining)

    previous_year = row.year
    previous_date = row.date.date()

station_hourly['remaining_bikes_proxy_now'] = remaining_values
station_hourly['refill_needed_proxy_now'] = np.maximum(TARGET_FILL - station_hourly['remaining_bikes_proxy_now'], 0)

for lag in [1, 24]:
    station_hourly[f'outflow_now_lag{lag}'] = station_hourly.groupby('year')['outflow_now'].shift(lag)
    station_hourly[f'inflow_now_lag{lag}'] = station_hourly.groupby('year')['inflow_now'].shift(lag)

feature_cols = [
    'hour_sin', 'hour_cos', 'month', 'dayofweek', 'is_weekend', 'is_holiday',
    'outflow_now', 'inflow_now', 'netflow_now',
    '기온(°C)', '습도(%)', '적설(cm)',
    'outflow_now_lag1', 'outflow_now_lag24',
    'inflow_now_lag1', 'inflow_now_lag24',
]

station_hourly['remaining_bikes_proxy_next'] = station_hourly.groupby('year')['remaining_bikes_proxy_now'].shift(-1)
station_hourly['refill_needed_proxy_next'] = station_hourly.groupby('year')['refill_needed_proxy_now'].shift(-1)

model_df = station_hourly.dropna(subset=feature_cols + ['remaining_bikes_proxy_next', 'refill_needed_proxy_next']).copy()
print('feature_count:', len(feature_cols))
print('feature_cols:', feature_cols)


feature_count: 16
feature_cols: ['hour_sin', 'hour_cos', 'month', 'dayofweek', 'is_weekend', 'is_holiday', 'outflow_now', 'inflow_now', 'netflow_now', '기온(°C)', '습도(%)', '적설(cm)', 'outflow_now_lag1', 'outflow_now_lag24', 'inflow_now_lag1', 'inflow_now_lag24']


In [5]:
train_valid_2024 = model_df[model_df['year'] == 2024].sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(train_valid_2024) * (1 - VALID_RATIO))
train_df = train_valid_2024.iloc[:split_idx].copy()
valid_df = train_valid_2024.iloc[split_idx:].copy()
test_df = model_df[model_df['year'] == 2025].copy()

X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]
X_test = test_df[feature_cols]

def evaluate_split(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R2': r2_score(y_true, y_pred),
    }


In [6]:
params = {
    'learning_rate': 0.05,
    'max_depth': 4,
    'max_iter': 300,
    'min_samples_leaf': 20,
    'l2_regularization': 1.0,
    'random_state': 42,
}

results = []

for target_col in ['remaining_bikes_proxy_next', 'refill_needed_proxy_next']:
    model = HistGradientBoostingRegressor(**params)
    model.fit(X_train, train_df[target_col])

    train_pred = model.predict(X_train)
    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    results.append({'target': target_col, 'split': 'train_2024', **evaluate_split(train_df[target_col], train_pred)})
    results.append({'target': target_col, 'split': 'valid_2024', **evaluate_split(valid_df[target_col], valid_pred)})
    results.append({'target': target_col, 'split': 'test_2025', **evaluate_split(test_df[target_col], test_pred)})

results_df = pd.DataFrame(results)
print('params:', params)
results_df


params: {'learning_rate': 0.05, 'max_depth': 4, 'max_iter': 300, 'min_samples_leaf': 20, 'l2_regularization': 1.0, 'random_state': 42}


,target,split,MAE,RMSE,R2
0,remaining_bikes_proxy_next,train_2024,3.695829,4.747417,0.586887
1,remaining_bikes_proxy_next,valid_2024,4.758198,5.958290,0.418742
2,remaining_bikes_proxy_next,test_2025,4.263698,5.390698,0.404678
3,refill_needed_proxy_next,train_2024,2.674878,3.661167,0.570591
4,refill_needed_proxy_next,valid_2024,3.444949,4.593949,0.411413
5,refill_needed_proxy_next,test_2025,3.032169,4.113815,0.377577


In [7]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../assets/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

trained_models = {}

for target_col in ["remaining_bikes_proxy_next", "refill_needed_proxy_next"]:
    model = HistGradientBoostingRegressor(**params)
    model.fit(X_train, train_df[target_col])
    trained_models[target_col] = model

joblib.dump(trained_models, MODEL_DIR / "bike_models.joblib")
joblib.dump(feature_cols, MODEL_DIR / "bike_feature_cols.joblib")
joblib.dump(params, MODEL_DIR / "bike_model_params.joblib")

print("saved:")
print(MODEL_DIR / "bike_models.joblib")
print(MODEL_DIR / "bike_feature_cols.joblib")
print(MODEL_DIR / "bike_model_params.joblib")


saved:
..\assets\models\bike_models.joblib
..\assets\models\bike_feature_cols.joblib
..\assets\models\bike_model_params.joblib
